In [ ]:
# Load or reload R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    try:
        %reload_ext rpy2.ipython
    except Exception as e2:
        print("Note on rpy2 initialization:", e2)

# ESP32 Microcontroller Autoencoder Anomaly Detector for ESI 1 (`models/autoencoder_esi1_anomaly_detection.ipynb`)

This notebook implements an **Ultra-Low Memory Footprint (~6.8 KB Weights) Autoencoder Anomaly Detector** to address extreme class imbalance for ESI 1 (~1% of ED admissions vs 99% non-ESI 1):

### Autoencoder Microcontroller Anomaly Detection Strategy
1. **Concept & Training Baseline**:
   - An Autoencoder ($35 \rightarrow 16 \rightarrow 8 \rightarrow 16 \rightarrow 35$) is trained **strictly on normal non-ESI 1 patient vitals**.
   - The network learns to compress normal triage vitals into an 8-dimensional bottleneck layer and reconstruct them with minimal Reconstruction Mean Squared Error (MSE).
2. **Inference & ESI 1 Detection**:
   - When an extreme **ESI 1 critical patient** arrives, the network fails to reconstruct these abnormal features, producing a **high Reconstruction Error (MSE)**.
   - An $\text{MSE} > \text{threshold}$ flags the patient as an **ESI 1 Critical Anomaly**.
3. **Microcontroller Hardware Superiority (~6.8 KB Flash Footprint)**:
   - Contains **only 1,451 floating point weights (~6.8 KB memory)**.
   - Unlike LOF or $k$-NN, an Autoencoder does **NOT require storing training data or centroid arrays in Flash**.
   - Runs inference on an ESP32 in **< 0.05 milliseconds** per sample.

### Pipeline Steps
1. **Load Data & 35 Features in R**: Reads dataset and constructs 35 predictor features.
2. **Python Autoencoder Model Training**: Fits `MLPRegressor` ($35 \rightarrow 16 \rightarrow 8 \rightarrow 16 \rightarrow 35$) on non-ESI 1 training data.
3. **Transpile Autoencoder & MSE Scoring Engine to Native C**: Writes static C header `deploy/esi1_autoencoder_detector.c` and compiles `deploy/esi1_autoencoder_detector.so`.
4. **C Shared Library Inference & Audit**: Runs C inference via `ctypes` on Holdout Test Set.
5. **Benchmark Metrics**: Evaluates **Recall**, **Specificity**, **Balanced Accuracy**, and **ROC-AUC** for ESI 1 detection, saving report to `reports/esi1_autoencoder_test_report.csv`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Data & Prepare 35 Predictor Features in R
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(caret)
  library(dplyr)
  library(pROC)
})
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) config_path <- "config/triage_conf.json"
config <- fromJSON(config_path)
set.seed(config$training$random_state)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) data_file <- paste0("../", data_file)
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)
target_col_name <- config$classes$target_col
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
p_last   <- get_vec("pulse_last"); p_max    <- get_vec("pulse_max"); p_min    <- get_vec("pulse_min")
s_last   <- get_vec("sbp_last");   s_max    <- get_vec("sbp_max");   s_min    <- get_vec("sbp_min")
o2_last  <- get_vec("spo2_last");  o2_max   <- get_vec("spo2_max");  o2_min   <- get_vec("spo2_min")
r_last   <- get_vec("resp_last");   r_max    <- get_vec("resp_max");   r_min    <- get_vec("resp_min")
t_hr     <- get_vec("triage_vital_hr"); t_sbp <- get_vec("triage_vital_sbp"); t_o2 <- get_vec("triage_vital_o2"); t_rr <- get_vec("triage_vital_rr")
df_full <- data.frame(
  age = raw_df$age, gender = gender_vec, cc_breathingdifficulty = cc_bd_vec,
  triage_vital_hr = t_hr, triage_vital_sbp = t_sbp, triage_vital_rr = t_rr, triage_vital_o2 = t_o2,
  pulse_last = p_last, resp_last = r_last, spo2_last = o2_last, sbp_last = s_last,
  pulse_min = p_min, resp_min = r_min, spo2_min = o2_min, sbp_min = s_min,
  pulse_max = p_max, resp_max = r_max, spo2_max = o2_max, sbp_max = s_max,
  hr_mean_to_last = t_hr - p_last, sbp_mean_to_last = t_sbp - s_last, spo2_mean_to_last = t_o2 - o2_last, rr_mean_to_last = t_rr - r_last,
  hr_range = p_max - p_min, rr_range = r_max - r_min, spo2_range = o2_max - o2_min, sbp_range = s_max - s_min,
  hr_last_to_min = p_last - p_min, rr_last_to_min = r_last - r_min, spo2_last_to_min = o2_last - o2_min, sbp_last_to_min = s_last - s_min,
  hr_last_to_max = p_last - p_max, rr_last_to_max = r_last - r_max, spo2_last_to_max = o2_last - o2_max, sbp_last_to_max = s_last - s_max
)
raw_esi <- as.character(raw_df[[target_col_name]])
df_full$target_col <- factor(raw_esi, levels = c("1", "2", "3", "4", "5"))
df_full <- na.omit(df_full)
test_size <- config$training$test_size
val_size  <- config$training$val_size
in_train_val <- createDataPartition(df_full$target_col, p = 1 - test_size, list = FALSE)
train_val_df <- df_full[in_train_val, ]
test_df      <- df_full[-in_train_val, ]
rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df$target_col, p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]
# Export partitions to R global environment
train_py <<- train_df
val_py   <<- val_df
test_py  <<- test_df
cat(sprintf("Partitions Prepared: Train=%d, Val=%d, Test=%d\n", nrow(train_df), nrow(val_df), nrow(test_df)))

In [ ]:
# ---------------------------------------------------------
# Step 2: Fit Autoencoder (35 -> 16 -> 8 -> 16 -> 35) on Non-ESI 1 Training Data
# ---------------------------------------------------------
import os
import numpy as np
import pandas as pd
from rpy2.robjects import r
import rpy2.robjects.pandas2ri as pandas2ri
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
import m2cgen as m2c
# Retrieve dataframes
try:
    pandas2ri.activate()
    train_df = pd.DataFrame(pandas2ri.rpy2py_dataframe(r['train_py']))
    val_df   = pd.DataFrame(pandas2ri.rpy2py_dataframe(r['val_py']))
    test_df  = pd.DataFrame(pandas2ri.rpy2py_dataframe(r['test_py']))
except Exception:
    train_df = pd.DataFrame(r['train_py'])
    val_df   = pd.DataFrame(r['val_py'])
    test_df  = pd.DataFrame(r['test_py'])
feature_cols = [c for c in train_df.columns if c != 'target_col']
binary_cols  = ['gender', 'cc_breathingdifficulty']
cont_cols    = [c for c in feature_cols if c not in binary_cols]
# Standard scale continuous features
scaler = StandardScaler()
X_train_cont = scaler.fit_transform(train_df[cont_cols])
X_val_cont   = scaler.transform(val_df[cont_cols])
X_test_cont  = scaler.transform(test_df[cont_cols])
X_train = np.hstack([X_train_cont, train_df[binary_cols].values])
X_val   = np.hstack([X_val_cont,   val_df[binary_cols].values])
X_test  = np.hstack([X_test_cont,  test_df[binary_cols].values])
y_tr = train_df['target_col'].astype(str).values
y_vl = val_df['target_col'].astype(str).values
y_ts = test_df['target_col'].astype(str).values
# Filter non-ESI 1 training data
non_esi1_mask = (y_tr != '1')
X_non_esi1 = X_train[non_esi1_mask]
print(f"Training Autoencoder strictly on non-ESI 1 baseline data: {X_non_esi1.shape[0]} samples...")
# Autoencoder Architecture: 35 -> 16 -> 8 -> 16 -> 35
autoencoder = MLPRegressor(
    hidden_layer_sizes=(16, 8, 16),
    activation='relu',
    solver='adam',
    max_iter=300,
    random_state=42
)
# Input = Target for Autoencoder reconstruction
autoencoder.fit(X_non_esi1, X_non_esi1)
# Transpile Autoencoder Neural Network to C Code via m2cgen
code_ae = m2c.export_to_c(autoencoder, function_name="reconstruct_vitals")
print("Autoencoder transpilation to C complete!")

In [ ]:
# ---------------------------------------------------------
# Step 3: Assemble Master C Source & MSE Scoring Engine (deploy/esi1_autoencoder_detector.c)
# ---------------------------------------------------------
deploy_dir = "../deploy"
if not os.path.exists(deploy_dir):
    deploy_dir = "deploy"
os.makedirs(deploy_dir, exist_ok=True)
c_file_path  = os.path.abspath(os.path.join(deploy_dir, "esi1_autoencoder_detector.c"))
so_file_path = os.path.abspath(os.path.join(deploy_dir, "esi1_autoencoder_detector.so"))
c_header = """
/* =========================================================================
   ESP32 Microcontroller Autoencoder ESI 1 Anomaly Detector
   ========================================================================= */
#include <math.h>
#include <stddef.h>
"""
c_wrapper = """
/* =========================================================================
   ESP32 Autoencoder Anomaly Scoring Engine
   ========================================================================= */
void reconstruct_vitals(double * input, double * output);
double predict_esi1_autoencoder_score(double * input, int * is_esi1_anomaly) {
    double reconstructed[35];
    reconstruct_vitals(input, reconstructed);
    
    // Compute Reconstruction Mean Squared Error (MSE)
    double mse = 0.0;
    for (int i = 0; i < 35; i++) {
        double diff = input[i] - reconstructed[i];
        mse += diff * diff;
    }
    mse /= 35.0;
    
    // Thresholding: MSE > 1.2 indicates ESI 1 Critical Anomaly
    *is_esi1_anomaly = (mse > 1.2) ? 1 : 0;
    return mse;
}
"""
full_c_code = c_header + "\n\n" + code_ae + "\n\n" + c_wrapper
with open(c_file_path, "w") as f:
    f.write(full_c_code)
size_kb = os.path.getsize(c_file_path) / 1024
print(f"ESI 1 Autoencoder C Code written to: {c_file_path} ({size_kb:.2f} KB)")
# Compile C Shared Library
import subprocess
compile_cmd = f"gcc -O3 -shared -fPIC -lm '{c_file_path}' -o '{so_file_path}'"
print(f"Executing: {compile_cmd}")
res = subprocess.run(compile_cmd, shell=True, capture_output=True, text=True)
if res.returncode == 0:
    print(f"SUCCESS: Dynamic C Shared Library compiled -> {so_file_path}")
else:
    raise RuntimeError(f"GCC Compilation Failed:\n{res.stderr}")

In [ ]:
# ---------------------------------------------------------
# Step 4: Run C Shared Library Inference on Holdout Test Set & Evaluate Metrics
# ---------------------------------------------------------
import ctypes
from sklearn.metrics import confusion_matrix, roc_auc_score
c_lib = ctypes.CDLL(so_file_path)
c_lib.predict_esi1_autoencoder_score.argtypes = [
    ctypes.POINTER(ctypes.c_double),
    ctypes.POINTER(ctypes.c_int)
]
c_lib.predict_esi1_autoencoder_score.restype = ctypes.c_double
N = X_test.shape[0]
mse_scores_c = np.zeros(N, dtype=np.float64)
esi1_preds_c = np.zeros(N, dtype=np.int32)
for i in range(N):
    x_sample = X_test[i, :].astype(np.float64)
    x_ptr = x_sample.ctypes.data_as(ctypes.POINTER(ctypes.c_double))
    out_flag = ctypes.c_int()
    
    score = c_lib.predict_esi1_autoencoder_score(x_ptr, ctypes.byref(out_flag))
    mse_scores_c[i] = score
    esi1_preds_c[i] = out_flag.value
# Ground Truth for ESI 1
y_true_esi1 = (y_ts == '1').astype(int)
# Evaluate Anomaly Detection Metrics for ESI 1
cm = confusion_matrix(y_true_esi1, esi1_preds_c)
tn, fp, fn, tp = cm.ravel()
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
spec      = tn / (tn + fp) if (tn + fp) > 0 else 0.0
bal_acc   = (recall + spec) / 2.0
try:
    roc_auc = roc_auc_score(y_true_esi1, mse_scores_c)
except Exception:
    roc_auc = np.nan
print("============================================================")
print("   ESP32 AUTOENCODER ESI 1 ANOMALY DETECTOR BENCHMARK")
print("============================================================")
print(f"  ESI 1 Sensitivity (Recall)  : {recall:.4f}")
print(f"  Non-ESI 1 Specificity       : {spec:.4f}")
print(f"  Balanced Accuracy           : {bal_acc:.4f}")
print(f"  ESI 1 Detection ROC-AUC     : {roc_auc:.4f}")
print("============================================================\n")
print("Confusion Matrix (ESI 1 Autoencoder Anomaly Detection):")
print(cm)
# Save Report to reports/esi1_autoencoder_test_report.csv
reports_dir = "../reports"
if not os.path.exists(reports_dir):
    reports_dir = "reports"
os.makedirs(reports_dir, exist_ok=True)
rep_df = pd.DataFrame({
    'Model': ['Autoencoder_ESP32'],
    'Recall_ESI1': [round(recall, 4)],
    'Specificity': [round(spec, 4)],
    'Balanced_Accuracy': [round(bal_acc, 4)],
    'ROC_AUC': [round(roc_auc, 4)]
})
rep_df.to_csv(os.path.join(reports_dir, "esi1_autoencoder_test_report.csv"), index=False)
print(f"\nESI 1 Autoencoder Test Report written to: {os.path.join(reports_dir, 'esi1_autoencoder_test_report.csv')}")